# Pandas 02 - Pandas Intermediate

> **MLCourse · Data Science Foundations · 02_pandas**

You know how to build, inspect, and slice DataFrames (notebook 01). Now we
tackle what makes real analysis *real*: messy values, grouped summaries,
reshaping, and stitching tables together.

## What you'll learn

- Missing data: detecting it, explaining it, dropping and filling it wisely
  (including **group-wise imputation**)
- Finding and removing duplicates
- Fixing dtypes: `astype`, `pd.to_numeric(errors="coerce")`, `pd.to_datetime`
- The apply family (`map`, `apply`, elementwise `.map`) - and when NOT to use it
- `replace`, binning with `pd.cut` / `pd.qcut`
- GroupBy deep dive: split-apply-combine, named aggregation, `transform`,
  `filter`
- Pivot tables & crosstabs; `melt`/`pivot`/`stack`/`unstack` reshaping
- Combining frames: `concat`, the four flavors of `merge`, and `join`

In [1]:
# Setup cell - same safe-import pattern as notebook 01.
try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass  # running as a plain script

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option("display.max_columns", 120)


def _load(name, demo_builder):
    """Download a seaborn dataset if possible, else use tiny demo rows."""
    try:
        return sns.load_dataset(name)
    except Exception as e:
        print(f"[setup] '{name}' unavailable ({type(e).__name__}) -> using demo rows.")
        return demo_builder()


def _demo_titanic():
    """Miniature titanic stand-in (same shape as the real thing)."""
    return pd.DataFrame(
        {
            "survived": [0, 1, 1, 1, 0, 0, 1, 0, 1, 0],
            "pclass":   [3, 1, 1, 1, 3, 3, 1, 3, 2, 3],
            "sex":      ["male", "female", "female", "female",
                         "male", "male", "male", "male", "female", "female"],
            "age":      [22.0, 38.0, 35.0, 27.0, 35.0, np.nan, 54.0, 2.0, 58.0, np.nan],
            "sibsp":    [1, 1, 0, 1, 0, 0, 0, 4, 0, 0],      # siblings/spouses aboard
            "parch":    [0, 0, 0, 0, 0, 0, 1, 1, 0, 2],      # parents/children aboard
            "fare":     [7.25, 71.28, 53.10, 57.00, 8.05, 8.46, 51.86, 21.07, 26.55, 7.75],
            "embarked": ["S", "C", "S", "S", "S", "Q", "S", "S", None, "S"],
        }
    )


def _demo_tips():
    """Miniature tips stand-in."""
    return pd.DataFrame(
        {
            "total_bill": [16.99, 10.34, 21.01, 23.68, 25.29, 8.77, 26.88, 15.04, 17.92, 44.30],
            "tip":        [1.01, 1.66, 3.50, 3.31, 2.73, 2.00, 5.00, 2.42, 3.08, 6.00],
            "sex":        ["F", "M", "M", "M", "F", "M", "M", "M", "F", "M"],
            "day":        ["Sun", "Sun", "Sat", "Sat", "Fri", "Thur", "Sun", "Sat", "Sat", "Fri"],
            "time":       ["Dinner", "Dinner", "Dinner", "Dinner", "Lunch",
                           "Lunch", "Dinner", "Dinner", "Dinner", "Lunch"],
            "size":       [2, 3, 3, 2, 4, 2, 4, 2, 2, 6],
        }
    )


try:
    titanic = sns.load_dataset("titanic")
except Exception as e:
    print("Could not download dataset (offline?):", e)
    titanic = _demo_titanic()

try:
    tips = sns.load_dataset("tips")
except Exception as e:
    print("Could not download dataset (offline?):", e)
    tips = _demo_tips()

print("titanic:", titanic.shape, "| tips:", tips.shape)

titanic: (891, 15) | tips: (244, 7)


## 1. Missing data

### 1.1 Three flavors of "nothing"

| object    | comes from                          | notes                              |
|-----------|-------------------------------------|------------------------------------|
| `None`    | Python (lists of mixed objects)     | becomes NaN inside numeric Series  |
| `np.nan`  | NumPy float world                   | a float! `nan != nan` is True-ish  |
| `pd.NA`   | modern nullable dtypes (`Int64`)    | pandas' missing-value singleton    |

WHY care: you never test missingness with `==`; you use `isna()`.

In [2]:
mixed = pd.Series([1.5, None, np.nan, 3.0])   # None is coerced to np.nan here
print(mixed, "\n")

# ⚠️ Common pitfall: comparing to NaN directly NEVER works -- even NaN == NaN
# is False! Equality can't represent "unknown".
print("x == nan gives:", mixed == np.nan)

# The RIGHT tools: isna()/notna() work for None, NaN, and pd.NA alike.
print("\nisna:", mixed.isna().tolist())
print("notna:", mixed.notna().tolist())

0    1.5
1    NaN
2    NaN
3    3.0
dtype: float64 

x == nan gives: 0    False
1    False
2    False
3    False
dtype: bool

isna: [False, True, True, False]
notna: [True, False, False, True]


In [3]:
# Per-column missing counts -> percentage table, biggest gaps first.
miss_cnt = titanic.isna().sum()                       # True counts as 1 when summed
miss_pct = (miss_cnt / len(titanic) * 100).round(1)   # share of rows missing

missing_report = (
    pd.DataFrame({"n_missing": miss_cnt, "pct": miss_pct})
    .query("n_missing > 0")
    .sort_values("pct", ascending=False)
)
missing_report

,n_missing,pct
deck,688,77.2
age,177,19.9
embarked,2,0.2
embark_town,2,0.2


### 1.2 Why does data go missing?

Not all NaNs are equal - and the reason changes your fix:

- **Not recorded** (someone skipped a survey question) - often random.
- **Does not apply** ("number of children" for a child) - structural.
- **Lost in processing** (failed join, bad parse) - your own bug!

> ⚠️ Common pitfall: filling without thinking injects bias. A deck column
> that's 77% empty tells you *the information doesn't exist* - no clever
> number will resurrect it.

In [4]:
# dropna(): remove rows (or columns) containing missing values.
tiny = pd.DataFrame(
    {
        "a": [1.0, np.nan, 3.0, np.nan],
        "b": [4.0, 5.0, np.nan, np.nan],
        "c": [7.0, 8.0, 9.0, np.nan],
    },
    index=list("wxyz"),
)
print(tiny, "\n")

print(tiny.dropna(), "\n")            # how="any": ANY NaN in row -> gone
print(tiny.dropna(how="all"), "\n")   # only fully-empty rows die ('y' survives)

print(tiny.dropna(subset=["a"]), "\n")          # judge only column 'a'
print(tiny.dropna(thresh=2))                    # keep rows with >= 2 real values

# 💡 Pro tip: dropping COLUMNS with huge gap shares (e.g. deck at ~77%) is
# often better than dropping thousands of ROWS over them.

     a    b    c
w  1.0  4.0  7.0
x  NaN  5.0  8.0
y  3.0  NaN  9.0
z  NaN  NaN  NaN 

     a    b    c
w  1.0  4.0  7.0 

     a    b    c
w  1.0  4.0  7.0
x  NaN  5.0  8.0
y  3.0  NaN  9.0 

     a    b    c
w  1.0  4.0  7.0
y  3.0  NaN  9.0 

     a    b    c
w  1.0  4.0  7.0
x  NaN  5.0  8.0
y  3.0  NaN  9.0


In [5]:
# fillna(): plug holes with constants or statistics.
messy = pd.Series([1.0, np.nan, np.nan, 8.0])

print(messy.fillna(0), "\n")            # constant sentinel -- fine for flags

# For measurements, prefer median/mean -- but WHICH one?
skewed = pd.Series([10, 12, 11, 13, 11, 500])   # 500 is an outlier!
print("mean :", skewed.mean())                  # dragged way up by 500
print("median:", skewed.median())               # robust to outliers

# Rule of thumb: SKEWED data -> median; roughly symmetric -> mean;
# categorical -> mode (most frequent value).
print("\nmedian fill:", messy.fillna(messy.median()).tolist())

# ffill/bfill: propagate the last (or next) VALID value forward/backward --
# designed for ordered data like time series or logs.
ordered = pd.Series([3.0, np.nan, np.nan, 9.0])
print("\nffill:", ordered.ffill().tolist())
print("bfill:", ordered.bfill().tolist())

0    1.0
1    0.0
2    0.0
3    8.0
dtype: float64 

mean : 92.83333333333333
median: 11.5

median fill: [1.0, 4.5, 4.5, 8.0]

ffill: [3.0, 3.0, 3.0, 9.0]
bfill: [3.0, 9.0, 9.0, 9.0]


### 1.3 Group-wise imputation (the professional move)

Filling every missing age with ONE global median ignores structure. Ages in
1st class skew older than in 3rd; men/women differ too. So we compute the
median **within each (class, sex) group** and fill with that.

The tool: `groupby(...)['col'].transform('median')` - it returns a Series
aligned 1-to-1 with the original rows, exactly what `fillna` wants.

In [6]:
# Step 1: one group-median PER ROW (transform broadcasts group stats back).
group_median_age = titanic.groupby(["pclass", "sex"])["age"].transform("median")

# Step 2: fill ONLY the holes; observed values stay untouched.
titanic_imputed = titanic.assign(
    age_filled=titanic["age"].fillna(group_median_age)
)

# Sanity checks: nothing lost, holes plugged.
print("before:", titanic["age"].isna().sum(), "missing")
print("after :", titanic_imputed["age_filled"].isna().sum(), "missing")

print(titanic_imputed.loc[titanic["age"].isna(), ["pclass", "sex", "age_filled"]].head())

# 💡 Pro tip: in ML pipelines, compute the medians on TRAINING data only and
# reuse them on test data -- otherwise you leak future knowledge into training.

before: 177 missing
after : 0 missing
    pclass     sex  age_filled
5        3    male        25.0
17       2    male        30.0
19       3  female        21.5
26       3    male        25.0
28       3  female        21.5


## 2. Duplicates

WHAT: identical (or key-identical) rows. WHY remove: they double-count
customers, inflate aggregates, and break joins silently.

In [7]:
signups = pd.DataFrame(
    {
        "email":     ["a@x.com", "b@x.com", "a@x.com", "c@x.com", "a@x.com"],
        "plan":      ["free", "pro", "free", "free", "free"],
        "signed_at": pd.to_datetime(
            ["2024-01-01", "2024-01-02", "2024-01-05", "2024-01-03", "2024-01-04"]
        ),
    }
)

# duplicated() marks repeats AFTER the first occurrence (keep='first').
print("duplicate rows:", signups.duplicated().sum())
print(signups[signups.duplicated()], "\n")

# Drop exact duplicates across ALL columns:
deduped = signups.drop_duplicates()
print(deduped.shape, "after full-row dedupe\n")

# Business rule: keep only each person's LATEST signup.
# Sort so the newest row lands LAST within each email group, then keep='last'.
latest = (
    signups
    .sort_values("signed_at")
    .drop_duplicates(subset=["email"], keep="last")   # subset = identity key(s)
    .sort_index()                                     # restore pleasant order
)
latest

# 💡 Pro tip: keep=False marks EVERY copy of a duplicate (great for auditing
# before you delete anything).

duplicate rows: 0
Empty DataFrame
Columns: [email, plan, signed_at]
Index: [] 

(5, 3) after full-row dedupe



,email,plan,signed_at
1,b@x.com,pro,2024-01-02
2,a@x.com,free,2024-01-05
3,c@x.com,free,2024-01-03


## 3. Fixing dtypes

CSVs love to deliver numbers as strings, dates as strings, categories as
strings... Converting early makes everything downstream faster and safer.

- `object` = "bag of Python objects" - usually strings; slow and vague.
- `category` = small integer codes + lookup table - fast, memory-lean,
  and semantically honest for repeated labels.

In [8]:
# 3.1 Numbers trapped in strings -> pd.to_numeric with errors="coerce".
dirty = pd.Series(["45", " 67 ", "N/A", "89", "abc"])   # spaces AND garbage
cleaned = pd.to_numeric(dirty.str.strip(), errors="coerce")
# errors="coerce": unparseable values become NaN instead of raising.
print(cleaned, "\n")
print("coerced to NaN:", cleaned.isna().sum(), "values")

0    45.0
1    67.0
2     NaN
3    89.0
4     NaN
dtype: float64 

coerced to NaN: 2 values


In [9]:
# 3.2 Dates from strings -> pd.to_datetime.
dates_messy = pd.Series(["2024-03-01", "01/04/2024", "not a date"])
parsed = pd.to_datetime(dates_messy, errors="coerce", format="mixed")
# format="mixed" lets pandas parse entries individually (pandas >= 2.0);
# for a single known layout pass it explicitly, e.g. format="%d/%m/%Y".
print(parsed, "\n")

# Explicit format = fastest AND unambiguous (is 01/04 April 1st or Jan 4th?).
eu_style = pd.Series(["01/04/2024", "25/12/2024"])       # day/month/year
sure = pd.to_datetime(eu_style, format="%d/%m/%Y")
print(sure.tolist())

0   2024-03-01
1   2024-01-04
2          NaT
dtype: datetime64[us] 

[Timestamp('2024-04-01 00:00:00'), Timestamp('2024-12-25 00:00:00')]


In [10]:
# 3.3 Strings -> category dtype.
before_mb = titanic["sex"].memory_usage(deep=True)
titanic_cat = titanic.assign(sex=titanic["sex"].astype("category"))
after_mb = titanic_cat["sex"].memory_usage(deep=True)
print(f"object: {before_mb:,} bytes -> category: {after_mb:,} bytes "
      f"({after_mb / before_mb:.0%} of original)")

# Full treatment of categorical SUPERPOWERS (ordering, categories mgmt):
# see 03_pandas_advanced.nb.py.

object: 55,111 bytes -> category: 1,147 bytes (2% of original)


## 4. The apply family - power tools with a safety label

Decision ladder, fastest first:

1. **Vectorized math** (`col * 2`, `col.sum()`)
2. **Accessor suites** (`.str.*`, `.dt.*`)
3. **`map`** with a dict/function (elementwise on one Series)
4. **`apply`** (row/column functions - flexible but slow)
5. Plain Python loop (last resort)

### 4.1 `map` on a Series

In [11]:
# map(dict): translate each value through a lookup table.
port_names = {"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"}
named_ports = titanic["embarked"].map(port_names)
print(named_ports.value_counts(dropna=False), "\n")

# map(function): transform each element.
name_lengths = titanic["sex"].map(len)     # length of each string
print(name_lengths.head(3).tolist())

# ⚠️ map(dict) turns UNMAPPED values into NaN -- verify coverage first!
odd_codes = pd.Series(["S", "C", "X"])     # 'X' isn't in port_names
print(odd_codes.map(port_names).tolist())

embarked
Southampton    644
Cherbourg      168
Queenstown      77
NaN              2
Name: count, dtype: int64 

[4, 6, 6]
['Southampton', 'Cherbourg', nan]


### 4.2 Why "vectorize first"? Let's measure.

In [12]:
import time as _time   # local helper import for this benchmark only

big = pd.Series(np.arange(1_000_001))                  # ~one million numbers

t0 = _time.perf_counter()                              # high-resolution timer
vec_result = big * 2                                   # vectorized: C speed
t_vec = _time.perf_counter() - t0

t0 = _time.perf_counter()
apply_result = big.apply(lambda x: x * 2)              # Python call PER CELL!
t_app = _time.perf_counter() - t0

print(f"vectorized : {t_vec:.4f}s")
print(f".apply     : {t_app:.4f}s  -> about {t_app / max(t_vec, 1e-9):.0f}x slower")

# Same answers, wildly different cost. Reach for apply only when NO vectorized
# equivalent exists (custom business logic per row, complex conditionals...).

vectorized : 0.0022s
.apply     : 0.3004s  -> about 137x slower


In [13]:
# 4.3 apply on Series vs DataFrame.
series_out = titanic["fare"].apply(lambda f: "expensive" if f > 100 else "normal")
print(series_out.value_counts(), "\n")

# DataFrame.apply works on WHOLE columns (axis=0, default) or rows (axis=1):
mini = pd.DataFrame({"math": [80, 55], "sci": [90, 65]})
print(mini.apply(sum), "\n")                    # axis=0: one sum PER COLUMN
print(mini.apply(lambda row: row.max() - row.min(), axis=1))

# Elementwise-on-every-cell used to be df.applymap(...); that name is DEPRECATED
# (removed in recent pandas). Use df.map(...) -- yes, same name as Series.map:
print(mini.map(lambda v: v / 100))

fare
normal       838
expensive     53
Name: count, dtype: int64 

math    135
sci     155
dtype: int64 

0    10
1    10
dtype: int64
   math   sci
0  0.80  0.90
1  0.55  0.65


> ⚠️ **Common pitfall:** reaching for `.apply` out of habit. Before writing
> one, ask: "can this be expressed as arithmetic, `.where`, or `.str`?"
> Your future self (and your CPU) will be grateful.

## 5. `replace`: surgical value swaps

In [14]:
# replace() differs from map(): UNTOUCHED values pass through unchanged
# (map would have NaN-ed them), making replace ideal for spot fixes.
codes = pd.Series(["S", "C", "Q", "S", "Z"])
fixed = codes.replace({"S": "Southampton", "C": "Cherbourg"})   # 'Q','Z' survive
print(fixed.tolist())

# List->list form replaces positionally: ['bad', 'awful'] -> ['poor', 'poor'].
grades = pd.Series(["good", "bad", "awful"])
print(grades.replace(["bad", "awful"], ["poor", "poor"]).tolist())

['Southampton', 'Cherbourg', 'Q', 'Southampton', 'Z']
['good', 'poor', 'poor']


## 6. Binning: continuous -> categories

WHAT: cut a numeric range into buckets. WHY: histograms, age bands, fare
classes - humans reason in categories, models sometimes need them too.

In [15]:
# pd.cut = EQUAL-WIDTH bins YOU define by edges (plus labels).
age_bands = pd.cut(
    titanic["age"],
    bins=[0, 12, 18, 35, 60, 200],           # edges; right edge inclusive by default
    labels=["child", "teen", "young_adult", "adult", "senior"],
    include_lowest=True,                     # make 0 itself included in bin 1
)
print(age_bands.value_counts(sort=False), "\n")   # sort=False keeps band order

# pd.qcut = QUANTILE bins: each bucket gets ~equal COUNTS.
fare_class = pd.qcut(
    titanic["fare"],
    q=4,                                      # quartiles
    labels=["budget", "economy", "premium", "luxury"],
)
print(fare_class.value_counts())

# ⚠️ Common pitfall: qcut fails with "duplicate edges" when many identical
# values sit at a boundary (e.g. hundreds of fares == 8.05). Remedy:
# qcut(..., duplicates="drop") and accept fewer buckets.

age
child           69
teen            70
young_adult    358
adult          195
senior          22
Name: count, dtype: int64 

fare
economy    224
budget     223
premium    222
luxury     222
Name: count, dtype: int64


## 7. GroupBy deep dive

The mental model - **split → apply → combine**:

```
split    : { class==1: rows..., class==2: rows..., class==3: rows... }
apply    : mean(fare) computed INSIDE each bucket
combine  : one result row per bucket, stitched back together
```

Every groupby call you will ever write is a variation on this theme.

In [16]:
# Watching split-apply-combine happen live: iterate groups explicitly
# (fine for exploration; avoid loops in production pipelines).
for class_id, group_df in titanic.groupby("pclass"):
    print(f"class {class_id}: {len(group_df):4d} passengers, "
          f"avg fare ${group_df['fare'].mean():.2f}")

class 1:  216 passengers, avg fare $84.15
class 2:  184 passengers, avg fare $20.66
class 3:  491 passengers, avg fare $13.68


In [17]:
# agg with a DICT: different stats for different columns.
summary_dict = titanic.groupby("pclass").agg(
    {"fare": ["mean", "max"], "age": "median"}
)
print(summary_dict, "\n")

# NAMED aggregation -- the modern, readable syntax:
#   new_name=("source_col", "aggfunc")
summary_named = titanic.groupby("pclass").agg(
    avg_fare=("fare", "mean"),
    top_fare=("fare", "max"),
    median_age=("age", "median"),
    n=("fare", "count"),          # count ignores NaN -- true sample sizes
).round(2)
print(summary_named)

# 💡 Pro tip: results are flat (single-level columns), ready to merge/export --
# one reason named aggregation beats the dict form for reports.

             fare              age
             mean       max median
pclass                            
1       84.154687  512.3292   37.0
2       20.662183   73.5000   29.0
3       13.675550   69.5500   24.0 

        avg_fare  top_fare  median_age    n
pclass                                     
1          84.15    512.33        37.0  216
2          20.66     73.50        29.0  184
3          13.68     69.55        24.0  491


In [18]:
# transform: SAME-SHAPE output as input -- group stats broadcast back to rows.
# This is THE tool for "compare each row to its group".
fare_z = (
    titanic["fare"]
    - titanic.groupby("pclass")["fare"].transform("mean")   # group center
) / titanic.groupby("pclass")["fare"].transform("std")      # group spread

standardized = titanic.assign(fare_z_within_class=fare_z.round(2))
# Interpretation: fare_z = +2 means "paid 2 standard deviations MORE than
# others in the same class".
print(standardized.nlargest(3, "fare_z_within_class")[["pclass", "fare", "fare_z_within_class"]])

# (We already used transform for group-wise AGE imputation in section 1.3.)

     pclass      fare  fare_z_within_class
258       1  512.3292                 5.46
679       1  512.3292                 5.46
737       1  512.3292                 5.46


In [19]:
# filter: keep/drop ENTIRE GROUPS by a group-level condition.
# Here: keep only weekdays whose average bill tops 20.
day_means = tips.groupby("day")["total_bill"].mean()
print(day_means.round(2), "\n")

busy_days = tips.groupby("day").filter(lambda g: g["total_bill"].mean() > 20)
kept_days = busy_days["day"].unique().tolist()
print("days surviving the filter:", sorted(kept_days))
print("rows before/after:", len(tips), "->", len(busy_days))

# 💡 Pro tip: `observed=True` in groupby controls whether unused CATEGORY
# levels appear; pandas 3 defaults sensibly, older versions needed the flag.
# Note: heavy custom logic still has groupby.apply as an escape hatch, but its
# API changed around pandas 2.2+ (grouping columns excluded) -- prefer
# agg/transform/filter whenever possible.

day
Thur    17.68
Fri     17.15
Sat     20.44
Sun     21.41
Name: total_bill, dtype: float64 



days surviving the filter: ['Sat', 'Sun']
rows before/after: 244 -> 163


## 8. Grouped pivoting: `pivot_table` and `crosstab`

WHAT: groupby + reshape into a spreadsheet-style grid.
WHY:  "average X per (A × B)" reads best as a matrix, not a long list.

In [20]:
# Survival rate by sex (rows) and class (columns).
survival_grid = pd.pivot_table(
    titanic,
    index="sex",              # becomes row labels
    columns="pclass",         # becomes column labels
    values="survived",        # what to aggregate
    aggfunc="mean",           # survival is 0/1 -> mean = rate
    margins=True,             # add All-rows / All-columns totals
    margins_name="All",
)
print(survival_grid.round(3), "\n")

# Median spend, filling empty combinations explicitly:
fare_grid = pd.pivot_table(
    titanic, index="embarked", columns="pclass",
    values="fare", aggfunc="median", fill_value=0, margins=True,
)
print(fare_grid.round(1))

pclass      1      2      3    All
sex                               
female  0.968  0.921  0.500  0.742
male    0.369  0.157  0.135  0.189
All     0.630  0.473  0.242  0.384

 

pclass       1     2    3   All
embarked                       
C         78.3  24.0  7.9  29.7
Q         90.0  12.4  7.8   7.8
S         52.0  13.5  8.0  13.0
All       58.7  14.2  8.0  14.5


In [21]:
# crosstab = frequency-table shortcut (counts by default).
counts = pd.crosstab(titanic["embarked"], titanic["pclass"])
print(counts, "\n")

# normalize='index': convert each ROW to percentages (rows sum to 1.0).
row_pct = pd.crosstab(titanic["embarked"], titanic["pclass"], normalize="index")
print(row_pct.round(3))

# 💡 pivot_table vs crosstab: pivot_table takes any aggfunc on any values;
# crosstab specializes in co-occurrence COUNTS (and their normalization).

pclass      1    2    3
embarked               
C          85   17   66
Q           2    3   72
S         127  164  353 

pclass        1      2      3
embarked                     
C         0.506  0.101  0.393
Q         0.026  0.039  0.935
S         0.197  0.255  0.548


## 9. Reshaping: `melt`, `pivot`, `stack`, `unstack`

Two shapes exist: **wide** (measurements as columns) and **long/tidy**
(one observation per row, variable names as DATA). Analysis prefers long;
presentation prefers wide; you must convert fluently.

In [22]:
grades_wide = pd.DataFrame(
    {
        "student": ["Ann", "Ben"],
        "math":    [88, 72],
        "science": [91, 85],
    }
)
print(grades_wide, "\n")

# melt: unpivot measurement columns into (variable, value) rows.
grades_long = grades_wide.melt(
    id_vars="student",                 # columns to KEEP as identifiers
    var_name="subject",                # new column holding former col names
    value_name="score",                # new column holding former cell values
)
print(grades_long)

  student  math  science
0     Ann    88       91
1     Ben    72       85 

  student  subject  score
0     Ann     math     88
1     Ben     math     72
2     Ann  science     91
3     Ben  science     85


In [23]:
# pivot: the inverse operation -- long back to wide.
back_to_wide = grades_long.pivot(
    index="student", columns="subject", values="score"
).reset_index()                        # student returns to a normal column
print(back_to_wide)

# ⚠️ Common pitfall: pivot REJECTS duplicate (index, columns) pairs. When
# duplicates are possible, use pivot_table (it aggregates them away).

subject student  math  science
0           Ann    88       91
1           Ben    72       85


In [24]:
# stack/unstack: the same dance driven by the INDEX.
# groupby with TWO keys produces a MultiIndex Series:
avg_fare = titanic.groupby(["pclass", "sex"])["fare"].mean()
print(avg_fare, "\n")

wide_fares = avg_fare.unstack()        # inner level (sex) -> columns
print(wide_fares.round(2), "\n")

long_again = wide_fares.stack()        # ...and back down to a tall series
print(long_again.head(4))

# 💡 stack/unstack shine after multi-key groupbys; pivot/melt shine on plain
# labeled columns. Choose whichever matches where your labels currently live.

pclass  sex   
1       female    106.125798
        male       67.226127
2       female     21.970121
        male       19.741782
3       female     16.118810
        male       12.661633
Name: fare, dtype: float64 

sex     female   male
pclass               
1       106.13  67.23
2        21.97  19.74
3        16.12  12.66 

pclass  sex   
1       female    106.125798
        male       67.226127
2       female     21.970121
        male       19.741782
dtype: float64


## 10. Combining DataFrames

### 10.1 `concat` - glue along an axis

In [25]:
first_half = tips.iloc[:5]     # positional slice: rows 0-4
second_half = tips.iloc[-3:]   # rows n-3..n-1

# Vertical stack (axis=0 default). ignore_index rebuilds 0..n-1 labels --
# otherwise the ORIGINAL indexes collide (0..4 next to n-3..n-1).
stacked = pd.concat([first_half, second_half], ignore_index=True)
print(stacked.shape, "<- 5 + 3 rows glued\n")

# keys= builds a MultiIndex tagging WHERE each piece came from:
tagged = pd.concat([first_half, second_half], keys=["early", "late"])
print(tagged.head(3), "\n")

# Horizontal concat (axis=1) side-by-side. Mismatched columns become NaN --
# unless you demand strictness with join="inner" (intersection only).
left = pd.DataFrame({"a": [1, 2], "shared": [10, 20]})
right = pd.DataFrame({"b": [99, 77], "shared": [10, 22]})
print(pd.concat([left, right], axis=1), "\n")                     # outer: union of cols
print(pd.concat([left, right], axis=1, join="inner"))             # keeps rows where ALL frames align

(8, 7) <- 5 + 3 rows glued

         total_bill   tip     sex smoker  day    time  size
early 0       16.99  1.01  Female     No  Sun  Dinner     2
      1       10.34  1.66    Male     No  Sun  Dinner     3
      2       21.01  3.50    Male     No  Sun  Dinner     3 

   a  shared   b  shared
0  1      10  99      10
1  2      20  77      22 

   a  shared   b  shared
0  1      10  99      10
1  2      20  77      22


### 10.2 `merge` - relational joins, the pandas way

| `how=`    | keeps...                                          |
|-----------|---------------------------------------------------|
| `"inner"` | only keys present in BOTH sides                   |
| `"left"`  | all left rows; right side NaN-filled when unmatched |
| `"right"` | mirror image of `"left"`                          |
| `"outer"` | union of keys; unmatched sides NaN-filled         |

We'll simulate a classic scenario: ORDERS referencing CUSTOMERS via a key.

In [26]:
customers = pd.DataFrame(
    {
        "cust_id": [1, 2, 3, 4],
        "name":    ["Ann", "Ben", "Cal", "Dee"],
        "city":    ["Rome", "Oslo", "Lima", "Kyoto"],
    }
)
orders = pd.DataFrame(
    {
        "order_id": [101, 102, 103, 104, 105, 106],
        "cust_id":  [1, 2, 2, 9, 3, 1],        # 9 = ghost customer, 4 = no orders
        "amount":   [250, 80, 120, 300, 40, 60],
        "city":     ["web", "web", "store", "web", "phone", "store"],  # overlaps!
    }
)

# INNER: order 104 (ghost id 9) disappears; customer Dee vanishes too.
inner = orders.merge(customers, on="cust_id", how="inner")
print("inner:", len(inner), "orders matched\n")

# LEFT: every order survives; the ghost's customer fields become NaN.
left_joined = orders.merge(customers, on="cust_id", how="left")
print(left_joined[left_joined["name"].isna()], "\n")

# Both tables had a 'city' column -> pandas auto-suffixed them.
print(list(left_joined.columns))

# suffixes= renames the collision explicitly:
labeled = orders.merge(customers, on="cust_id", how="left",
                       suffixes=("_order", "_customer"))
print(list(labeled.columns)[-4:], "<- explicit suffixes")

inner: 5 orders matched

   order_id  cust_id  amount city_x name city_y
3       104        9     300    web  NaN    NaN 

['order_id', 'cust_id', 'amount', 'city_x', 'name', 'city_y']
['amount', 'city_order', 'name', 'city_customer'] <- explicit suffixes


In [27]:
# indicator=True adds a provenance column -- perfect for audits:
audited = orders.merge(customers, on="cust_id", how="outer",
                       indicator=True, suffixes=("_o", "_c"))
print(audited["_merge"].value_counts(), "\n")
print(audited[audited["_merge"] == "right_only"][["cust_id", "name"]])  # Dee!

# validate= asserts the relationship cardinality -- cheap insurance:
many_to_one = orders.merge(customers, on="cust_id", how="left",
                           validate="many_to_one")   # would RAISE on a bad key
print("validated merge ok:", len(many_to_one), "rows")
# Other contracts: "one_to_one", "one_to_many", "many_to_many".

_merge
both          5
left_only     1
right_only    1
Name: count, dtype: int64 

   cust_id name
5        4  Dee
validated merge ok: 6 rows


In [28]:
# Business question: revenue per customer INCLUDING customers with zero orders.
revenue = (
    orders
    .merge(customers, on="cust_id", how="left")            # keep ghost orders visible
    .assign(name=lambda d: d["name"].fillna("(unknown)"))  # label the ghosts
    .groupby("name", as_index=False)["amount"].sum()
    .sort_values("amount", ascending=False)
)
print(revenue)

# Dee shows zero orders because SHE never appears in orders -- flip the merge
# (customers LEFT-join orders) when the customer list must drive the report:
per_customer = (
    customers
    .merge(orders, on="cust_id", how="left", suffixes=("", "_order"))
    .groupby("name")["amount"].sum().fillna(0)             # NaN sum -> 0.0
)
print(per_customer)

        name  amount
1        Ann     310
0  (unknown)     300
2        Ben     200
3        Cal      40
name
Ann    310.0
Ben    200.0
Cal     40.0
Dee      0.0
Name: amount, dtype: float64


In [29]:
# Different column NAMES on each side? Use left_on/right_on.
staff = pd.DataFrame({"person_id": [1, 2], "who": ["Eve", "Finn"]})
shifts = pd.DataFrame({"worker": [2, 1], "hours": [6, 8]})
linked = shifts.merge(staff, left_on="worker", right_on="person_id")
print(linked[["who", "hours"]], "\n")

# join() = merge pre-wired to INDEXES; with on= it joins a column to an index.
# (drop orders' overlapping 'city' first -- join has no auto-suffixing like merge)
by_index_join = orders.join(
    customers.drop(columns="city").set_index("cust_id"), on="cust_id"
)
print("join(on=...) equals a left merge:",
      by_index_join["name"].isna().sum() == left_joined["name"].isna().sum())

    who  hours
0  Finn      6
1   Eve      8 

join(on=...) equals a left merge: True


## Summary & key takeaways

- Detect missingness with `isna()` (never `== nan`); build a count/pct
  report before deciding anything.
- Drop with intent (`how=`, `subset=`, `thresh=`); fill by statistic chosen
  by skew (median for skewed, mean for symmetric, mode for categories);
  `ffill/bfill` for ordered data; **group-by-group medians beat one global
  median** (`groupby + transform`).
- Deduplicate with `duplicated()` then `drop_duplicates(subset, keep=)`.
- Convert dtypes aggressively-but-carefully: `astype("category")`,
  `pd.to_numeric(errors="coerce")`, `pd.to_datetime(format=...)`.
- Vectorize first; `map` for lookups; `apply` only when nothing faster fits;
  `df.applymap` is retired - use `df.map`.
- `pd.cut` for fixed-width bins, `pd.qcut` for equal-count bins.
- GroupBy = split-apply-combine; named aggregation (`avg=("fare","mean")`)
  for readable summaries; `transform` for row-aligned group stats; `filter`
  for keeping whole groups.
- Reshape freely: `melt`/`pivot` between long and wide, `stack`/`unstack`
  around MultiIndexes, `pivot_table`/`crosstab` for instant matrices.
- `concat` stacks; `merge(how=...)` joins relationally - lean on
  `indicator=True` and `validate=` to stay honest.

**Next up:** `03_pandas_advanced.nb.py` - time series, MultiIndex mastery,
string superpowers, Copy-on-Write, and memory tuning.